## Задание 1
Прочитать датасет.

In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv('StudentsPerformanceV3.tsv', sep='\t')
df

,student id,date,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,SF1,2002-3-15,female,group B,bachelor's degree,standard,none,72,72,74
1,SF2,2004-7-13,female,group C,some college,standard,completed,69,90,88
2,SF3,2004-8-23,female,group B,master's degree,standard,none,90,95,93
3,SM4,2000-6-3,male,group A,associate's degree,free/reduced,none,47,57,44
4,SM5,1999-9-6,male,group C,some college,standard,none,76,78,75
...,...,...,...,...,...,...,...,...,...,...
1011,SF392,1998-6-26,female,group D,some college,standard,none,72,68,77
1012,SM393,1998-6-1,male,group E,some college,standard,none,76,71,62
1013,SM869,2004-6-15,male,group E,associate's degree,free/reduced,completed,70,74,70
1014,SM870,1999-3-19,male,group C,associate's degree,free/reduced,none,49,52,51


In [3]:
df.isna().sum()

student id                      6
date                            3
gender                          9
race/ethnicity                  7
parental level of education     7
lunch                           6
test preparation course         6
math score                      9
reading score                  10
writing score                   9
dtype: int64

## Задание 2
Привести math score, reading score, writing score к числовому виду: там, где вместо числа стоит буква (A/B/C/D/E/F), преобразовать по шкале A=100, B=80, C=60, D=40, E=20, F=0, используя собственную функцию и .apply().

In [4]:
def convert_score(value):
    letters_map = {
        'A': 100,
        'B': 80,
        'C': 60,
        'D': 40,
        'E': 20,
        'F': 0}

    if isinstance(value, str): # проверяем является ли value строкой
        value = value.strip()
        if value in letters_map:
            return letters_map[value]
        try:
            return float(value)
        except ValueError:
            return np.nan
    return value

In [5]:
columns = ['math score', 'reading score', 'writing score']

for col in columns:
    df[col] = df[col].apply(convert_score)

df['math score'].unique() # проверяем поменялись ли буквы на цифры

array([ 72.,  69.,  90.,  47.,  76.,  71.,  88.,  40.,  64.,  38.,  58.,
        65.,  78.,  50.,  18.,  46.,  54.,  66.,  44.,  74.,  73.,  67.,
        70.,  62.,  63.,  56.,  97.,  81.,  75.,  57.,  55.,  53.,  83.,
        87.,  nan,  59.,  82.,  77.,  33., 100.,   0.,  79.,  39.,  45.,
        60.,  61.,  41.,  49.,  30.,  80.,  42.,  27.,  43.,  68.,  85.,
        98.,  52.,  51.,  99.,  84.,  91.,  89.,  22.,  96.,  94.,  48.,
        35.,  34.,  86.,  92.,  20.,  37.,  28.,  24.,  26.,  95.,  36.,
        29.,  32.,  93.,  19.,  23.,   8.])

## Задание 3
Удалить строки, где абсолютно все значения пропущены.

In [6]:
df = df.dropna(how='all')
df.isna().sum()

student id                     3
date                           0
gender                         6
race/ethnicity                 4
parental level of education    4
lunch                          3
test preparation course        3
math score                     6
reading score                  7
writing score                  6
dtype: int64

## Задание 4
Удалить строки без student id — такой id восстановить нечем.

In [7]:
df = df.dropna(subset=['student id'])
df.isna().sum()

student id                     0
date                           0
gender                         3
race/ethnicity                 1
parental level of education    1
lunch                          0
test preparation course        0
math score                     3
reading score                  4
writing score                  3
dtype: int64

## Задание 5
Заполнить пропуски в race/ethnicity, parental level of education, math score на основе похожих записей — отфильтровать студентов с теми же остальными демографическими признаками и взять .mode()/.mean() по этой подвыборке.

Заполнение race/ethnicity

In [8]:
def fill_race(row):
    if pd.isna(row['race/ethnicity']):
        similar = df[
            (df['gender'] == row['gender']) &
            (df['parental level of education'] == row['parental level of education']) &
            (df['lunch'] == row['lunch']) &
            (df['test preparation course'] == row['test preparation course']) &
            (df['race/ethnicity'].notna())
        ]

        if not similar.empty:
            return similar['race/ethnicity'].mode()[0]

    return row['race/ethnicity']

In [9]:
df['race/ethnicity'] = df.apply(fill_race, axis=1)
df.isna().sum()

student id                     0
date                           0
gender                         3
race/ethnicity                 0
parental level of education    1
lunch                          0
test preparation course        0
math score                     3
reading score                  4
writing score                  3
dtype: int64

Заполнение parental level of education

In [10]:
def fill_parental_education(row):
    if pd.isna(row['parental level of education']):
        similar = df[
            (df['gender'] == row['gender']) &
            (df['race/ethnicity'] == row['race/ethnicity']) &
            (df['lunch'] == row['lunch']) &
            (df['test preparation course'] == row['test preparation course']) &
            (df['parental level of education'].notna())
        ]

        if not similar.empty:
            return similar['parental level of education'].mode()[0]

    return row['parental level of education']

In [11]:
df['parental level of education'] = df.apply(fill_parental_education, axis=1)
df.isna().sum()

student id                     0
date                           0
gender                         3
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     3
reading score                  4
writing score                  3
dtype: int64

Заполнение math score. Так как пропуске в колонке с полом нужно заполнять в следующем задании, то в этом я сделаю вторую подгруппу без включения пола в нее

In [12]:
def fill_math_score(row):
    if pd.isna(row['math score']):
        similar = df[
            (df['gender'] == row['gender']) &
            (df['race/ethnicity'] == row['race/ethnicity']) &
            (df['parental level of education'] == row['parental level of education']) &
            (df['lunch'] == row['lunch']) &
            (df['test preparation course'] == row['test preparation course']) &
            (df['math score'].notna())
        ]
        
        if not similar.empty:
            return similar['math score'].mean()

        similar = df[
                    (df['race/ethnicity'] == row['race/ethnicity']) &
                    (df['parental level of education'] == row['parental level of education']) &
                    (df['lunch'] == row['lunch']) &
                    (df['test preparation course'] == row['test preparation course']) &
                    (df['math score'].notna())
                ]
                
        if not similar.empty:
            return similar['math score'].mean()
    
    return row['math score']

In [13]:
df['math score'] = df.apply(fill_math_score, axis=1)
df[df.isna().any(axis=1)]

,student id,date,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
7,SM8,2004-3-13,male,group B,some college,free/reduced,none,40.000000,NaN,39.0
107,SM101,2000-5-1,NaN,group B,some college,standard,none,60.916667,67.0,67.0
204,SM198,1998-9-6,male,group E,high school,free/reduced,none,55.000000,NaN,51.0
218,SM212,1998-2-11,male,group C,some college,free/reduced,none,35.000000,28.0,NaN
280,SF274,2004-5-15,NaN,group D,some college,standard,none,65.000000,70.0,71.0
287,SM281,1999-9-3,male,group D,high school,standard,none,53.000000,52.0,NaN
353,SM347,2002-1-3,NaN,group B,some college,standard,none,62.000000,61.0,57.0
645,SM639,2004-10-10,male,group E,some college,standard,none,86.000000,76.0,NaN
659,SF653,2003-11-19,female,group D,some college,standard,completed,85.000000,NaN,98.0
905,SM899,2004-5-12,male,group D,associate's degree,standard,completed,67.000000,NaN,63.0


In [14]:
df.isna().sum()

student id                     0
date                           0
gender                         3
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  4
writing score                  3
dtype: int64

## Задание 6
Заполнить оставшиеся пропуски в writing score и reading score статистикой (по подгруппе или по всему датасету — обоснуйте выбор).

Обоснование: Использовал оба подхода: если получилось собрать подгруппу с теми же признаками, то берем среднее значение из нее, иначе берем по всему датасету

In [15]:
group_cols = [
    'gender', 
    'race/ethnicity', 
    'parental level of education', 
    'lunch', 
    'test preparation course'
]

def fill_score(row, score_col):
    if pd.isna(row[score_col]):
        # Берем значение по подгруппе
        cond = pd.Series(True, index=df.index)
        for col in group_cols:
            cond = cond & (df[col] == row[col])
        
        similar = df[cond & df[score_col].notna()]
        if not similar.empty:
            return similar[score_col].mean()
        
        # Если подгруппа не получилась, то берем значение по всему датасету 
        return df[score_col].mean()
    
    return row[score_col]

df['reading score'] = df.apply(lambda row: fill_score(row, 'reading score'), axis=1)
df['writing score'] = df.apply(lambda row: fill_score(row, 'writing score'), axis=1)


df.isna().sum()


student id                     0
date                           0
gender                         3
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

## Задание 7
Восстановить пропуски в gender, используя структуру student id (.str) — во второй позиции id закодирован пол.

In [16]:
df[df.isna().any(axis=1)]

,student id,date,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
107,SM101,2000-5-1,NaN,group B,some college,standard,none,60.916667,67.0,67.0
280,SF274,2004-5-15,NaN,group D,some college,standard,none,65.000000,70.0,71.0
353,SM347,2002-1-3,NaN,group B,some college,standard,none,62.000000,61.0,57.0


In [17]:
gender_letter = df['student id'].str[1]
gender_map = {'M': 'male', 'F': 'female'}
final_gender = gender_letter.map(gender_map)

df['gender'] = df['gender'].fillna(final_gender)

df.isna().sum()

student id                     0
date                           0
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

## Задание 8
Обработать дубликаты по student id: если для одного id несколько записей, оставить самую свежую по колонке date. Даты нужно распарсить; среди них попадаются некорректные календарные даты (например, 30 февраля) — обработайте и это.

In [18]:
df1 = df[df.duplicated(subset=['student id'], keep=False)]
df1

,student id,date,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
8,SM9,2000-4-30,male,group D,high school,free/reduced,completed,64.0,64.0,67.0
9,SF10,2004-11-31,female,group B,high school,free/reduced,none,38.0,60.0,50.0
10,SM11,1999-10-16,male,group C,associate's degree,standard,none,58.0,54.0,52.0
313,SF309,2001-12-17,female,group B,associate's degree,free/reduced,none,53.0,71.0,67.0
315,SF309,1999-8-24,female,group B,associate's degree,free/reduced,none,50.0,72.0,83.0
397,SM391,1999-12-20,male,group E,some high school,free/reduced,completed,73.0,67.0,59.0
398,SF392,2001-10-2,female,group D,some college,standard,none,77.0,68.0,77.0
399,SM393,2002-5-15,male,group E,some college,standard,none,76.0,67.0,67.0
875,SM869,2001-6-19,male,group E,associate's degree,free/reduced,completed,78.0,74.0,72.0
876,SM870,2002-1-28,male,group C,associate's degree,free/reduced,none,49.0,51.0,51.0


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1010 entries, 0 to 1015
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   student id                   1010 non-null   object 
 1   date                         1010 non-null   object 
 2   gender                       1010 non-null   object 
 3   race/ethnicity               1010 non-null   object 
 4   parental level of education  1010 non-null   object 
 5   lunch                        1010 non-null   object 
 6   test preparation course      1010 non-null   object 
 7   math score                   1010 non-null   float64
 8   reading score                1010 non-null   float64
 9   writing score                1010 non-null   float64
dtypes: float64(3), object(7)
memory usage: 86.8+ KB


In [20]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20 entries, 8 to 1015
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   student id                   20 non-null     object 
 1   date                         20 non-null     object 
 2   gender                       20 non-null     object 
 3   race/ethnicity               20 non-null     object 
 4   parental level of education  20 non-null     object 
 5   lunch                        20 non-null     object 
 6   test preparation course      20 non-null     object 
 7   math score                   20 non-null     float64
 8   reading score                20 non-null     float64
 9   writing score                20 non-null     float64
dtypes: float64(3), object(7)
memory usage: 1.7+ KB


То есть должно получиться минус 10 дубликатов и итогово 1000 записей в датасете

In [21]:
df['getted date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.sort_values(by='getted date')
df = df.drop_duplicates(subset=['student id'], keep='last')
df = df.drop(columns=['getted date'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 431 to 985
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   student id                   1000 non-null   object 
 1   date                         1000 non-null   object 
 2   gender                       1000 non-null   object 
 3   race/ethnicity               1000 non-null   object 
 4   parental level of education  1000 non-null   object 
 5   lunch                        1000 non-null   object 
 6   test preparation course      1000 non-null   object 
 7   math score                   1000 non-null   float64
 8   reading score                1000 non-null   float64
 9   writing score                1000 non-null   float64
dtypes: float64(3), object(7)
memory usage: 85.9+ KB


## Задание 9 
Сохранение результата

In [22]:
df.to_csv('StudentsPerformanceV3Final.csv', index=False)